In [ ]:
# we are going to make project based on the parellel llm name is upsc essay evaluation 
from langgraph.graph import StateGraph
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field
import operator

ModuleNotFoundError: No module named 'langchain_openai'

In [ ]:
load_dotenv()

In [ ]:
model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
class evaluationSchema(BaseModel):
    feedback:str=Field(description="detailed feedback of the essay")
    score:int=Field(description="score out of the 10")

In [ ]:
structured_model=model.with_structured_output(evaluationSchema)

In [ ]:
essay="""

Artificial Intelligence (AI) has become an essential part of modern life. From smartphones and virtual assistants to online shopping and healthcare, AI is transforming the way people live, work, and communicate. It refers to the ability of machines and computer systems to perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and understanding language.

One of the most common examples of AI is the use of virtual assistants like Siri, Google Assistant, and Alexa. These systems can answer questions, set reminders, play music, and control smart home devices through voice commands. Similarly, recommendation systems used by platforms such as YouTube, Netflix, and online shopping websites analyze user preferences to suggest relevant content and products.

AI has also brought significant improvements to healthcare. Advanced algorithms help doctors diagnose diseases more accurately and at an earlier stage. AI-powered tools can analyze medical images, predict health risks, and assist in developing personalized treatment plans. During emergencies, AI can help hospitals manage resources efficiently and improve patient care.

In the field of education, AI enables personalized learning experiences. Educational platforms can adapt lessons according to a student's strengths and weaknesses, making learning more effective. Chatbots and virtual tutors provide instant assistance, allowing students to learn at their own pace.

Businesses use AI to automate repetitive tasks, improve customer service, and analyze large amounts of data. This helps organizations make better decisions and increase productivity. In transportation, AI plays a crucial role in navigation systems, traffic management, and the development of self-driving vehicles.

Despite its many advantages, AI also presents challenges. Concerns about data privacy, job displacement, and ethical use of technology need careful consideration. It is important to ensure that AI systems are developed responsibly and used in ways that benefit society as a whole.

In conclusion, Artificial Intelligence is revolutionizing everyday life by making services faster, smarter, and more efficient. As technology continues to evolve, AI will play an even greater role in shaping the future. By balancing innovation with ethical responsibility, society can harness the full potential of AI for the betterment of humanity.
"""

In [ ]:
prompt = f'Evaluate the language quality of the following essay and provide feedback and assign a score out of 10 \n {essay}'
# structured_model.invoke(prompt).score when you want score 
structured_model.invoke(prompt).feedback #when user want feedback they can run this code 

In [ ]:
class upscstate(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    indivisual_score:Annotated[list[int],operator.add]
    avg_score:float

In [ ]:
def evaluate_language(state:upscstate):
    prompt = f'Evaluate the language quality of the following essay and provide feedback and assign a score out of 10 \n state[essay]'
    output=structured_model.invoke(prompt)
    return {'language_feedback':output.feedback,'individual_scores':[output.score]}


In [ ]:
def evaluate_analysis(state:upscstate):
    prompt = f'Evaluate the  deep analysis of the following essay and provide feedback and assign a score out of 10 \n state[essay]'
    output=structured_model.invoke(prompt)
    return {'analysis_feedback':output.feedback,'individual_scores':[output.score]}

In [ ]:
def evaluate_thought(state:upscstate):
    prompt = f'Evaluate the deep thought and clarity of the following essay and provide feedback and assign a score out of 10 \n state[essay]'
    output=structured_model.invoke(prompt)
    return {'clarity_feedback':output.feedback,'individual_scores':[output.score]}

In [ ]:
graph=StateGraph(upscstate)
graph.add_nodes('evaluate_language',evaluate_language)
graph.add_nodes('evaluate_analysis',evaluate_analysis)
graph.add_nodes('evaluate_thought',evaluate_thought)``
graph.add_nodes('final_evalution',final_evalution)